In [1]:
import json
import pandas as pd
from tqdm import tqdm

In [2]:
ner_dset_filename = "dataset/99.인도네시아 말뭉치 데이터_sample_230823_v.02.json"
with open(ner_dset_filename, 'r') as data_file:
    json_data = data_file.read()
ner_dset = json.loads(json_data)

In [4]:
len(ner_dset[:3])

3

In [5]:
lang_id = "id"
inst_templates_filename = "dataset/instruction_templates.csv"
inst_templates = pd.read_csv(inst_templates_filename, names=['lang', 'input', 'output'], header=0)
inst_templates = inst_templates[inst_templates.lang == lang_id].reset_index().to_dict('records')

In [6]:
main_tag_to_name = {
    'PS': 'Person',
    'PD': 'Product',
    'WA': 'Work of Art',
    'BO': 'Brand & Organization',
    'LC': 'Location',
    'EV': 'Event',
}
def get_tag_name(bio_tag):
    tags = bio_tag.split("_")
    return f"{main_tag_to_name[tags[0]]} ({' '.join(tags[1:-1])})"


def bio_to_group(bio_tags, tokens):
    group_tags = []
    current_entity = []
    current_tag = "O"
    
    for token, bio_tag in zip(tokens, bio_tags):
        if bio_tag == "O":
            if current_entity:
                group_tags.append((" ".join(current_entity), current_tag))
                current_entity = []
                current_tag = "O"
        elif bio_tag.endswith("_B"):
            if current_entity:
                group_tags.append((" ".join(current_entity), current_tag))
            current_entity = [token]
            current_tag = get_tag_name(bio_tag)
        elif bio_tag.endswith("_I"):
            current_entity.append(token)
    
    if current_entity:
        group_tags.append((" ".join(current_entity), current_tag))
    
    return group_tags

In [7]:
def get_processed_entities(sent_data, ent_type=None):
    entities_text = ""
    entities_json = []
    group_tags = bio_to_group(sent_data['Entities'], sent_data['Raw_data'].split())
    if ent_type is not None:
        group_tags = [item for item in group_tags if item[1] == ent_type]

    for idx, item in enumerate(group_tags):
        if ent_type is None:
            if entities_text == "":
                entities_text = f"{idx+1}. {item[0]}: {item[1]}"
            else:
                entities_text += f"\n{idx+1}. {item[0]}: {item[1]}"
            entities_json.append({
                'token': item[0],
                'entity_type': item[1]
            })
        else:
            if entities_text == "":
                entities_text = f"{idx+1}. {item[0]}"
            else:
                entities_text += f"\n{idx+1}. {item[0]}"
            entities_json.append({
                'token': item[0],
                'entity_type': item[1]
            })

    if entities_text == "":
        if ent_type is not None:
            if lang_id == "id":
                entities_text = f"Entitas dengan tipe \"{ent_type}\" tidak ditemukan."
            else:
                entities_text = f"Entities with \"{ent_type}\" type not found."
        else:
            entities_text = "Entitas tidak ditemukan" if lang_id == "id" else "Entities not found."
        entities_json_text = entities_text
    else:
        entities_json_text = json.dumps(entities_json)

    return entities_text, entities_json_text

In [15]:
inst_dset = []
for item in tqdm(ner_dset[-1:]):
    for item_data in item['data']:
        for template in inst_templates:
            inst_id = f"{item_data['Sen_ID']}_{template['index']}"
            inst_prompt = template['input'].replace("{{Raw-Data}}", item_data['Raw_data'])
            inst_generation = template['output'].replace("{{Raw-Data}}", item_data['Raw_data'])

            if "{{Entity-Type}}" in template['input']:
                ent_text, ent_json = get_processed_entities(item_data, 'Location (Country)')
                inst_prompt = inst_prompt.replace("{{Entity-Type}}", 'Location (Country)').replace("{{Processed-Entities-Type-Only}}", ent_text)
                inst_generation = inst_generation.replace("{{Entity-Type}}", 'Location (Country)').replace("{{Processed-Entities-Type-Only}}", ent_text)
            else:
                ent_text, ent_json = get_processed_entities(item_data)
            
            inst_prompt = inst_prompt.replace("{{Processed-Entities}}", ent_text).replace("{{Processed-Entities-JSON}}", ent_json)
            inst_generation = inst_generation.replace("{{Processed-Entities}}", ent_text).replace("{{Processed-Entities-JSON}}", ent_json)

            if "tidak ditemukan" in inst_generation or "not found" in inst_generation:
                inst_generation = ent_text

            inst_dset.append({
                'id': inst_id,
                'prompt': inst_prompt,
                'generation': inst_generation
            })

100%|██████████| 1/1 [00:00<00:00, 966.88it/s]


In [17]:
len(inst_dset)

25

In [18]:
out_json_filename = "dataset/test_samples.jsonl"
with open(out_json_filename, 'w') as jsonl_file:
    for item in inst_dset:
        json.dump(item, jsonl_file)
        jsonl_file.write('\n')

In [18]:
inst_dset_df = pd.DataFrame(inst_dset)

In [19]:
inst_dset_df

,id,prompt,generation
0,20230808_newsdata_Korea_000366_000001_0,Please state the entities of the following tex...,The entities are:\n1. V HEARTBEAT: Event (Fest...
1,20230808_newsdata_Korea_000366_000001_2,V HEARTBEAT ' The Hateaway to Asia ' akhirnya ...,Entities:\n1. V HEARTBEAT: Event (Festival)\n2...
2,20230808_newsdata_Korea_000366_000001_4,List out the entities in the following text: V...,"[{""token"": ""V HEARTBEAT"", ""entity_type"": ""Even..."
3,20230808_newsdata_Korea_000366_000001_6,List out the words that have Person (Name) in ...,"Entities with ""Person (Name)"" type not found."
4,20230808_newsdata_Korea_000366_000001_8,Form a sentence based on the following entitie...,V HEARTBEAT ' The Hateaway to Asia ' akhirnya ...
...,...,...,...
165,20230808_newsdata_Korea_000395_000005_0,Please state the entities of the following tex...,The entities are:\n1. Kim Junsu: Person (Name)
166,20230808_newsdata_Korea_000395_000005_2,' Alasan mengapa pihak Kim Junsu merilis perny...,Entities:\n1. Kim Junsu: Person (Name)
167,20230808_newsdata_Korea_000395_000005_4,List out the entities in the following text: '...,"[{""token"": ""Kim Junsu"", ""entity_type"": ""Person..."
168,20230808_newsdata_Korea_000395_000005_6,List out the words that have Person (Name) in ...,Words with Person (Name) entity type: 1. Kim J...


In [20]:
inst_dset_df.to_csv(f"dataset/ner_{lang_id}_instruction_examples.csv", index=False)